# Conversational Travel Assistant — LangChain Live Build

**What we're building:** a small assistant for a demo travel company that can
look up weather, suggest attractions, and search flights — using three fake
local data sources — and that remembers what was said earlier in the
conversation.

**What this notebook assumes you already know:** basic LangChain usage
(prompts, a simple chain, calling a chat model). We're not re-explaining
those basics — we're explaining how tools, tool-calling, and memory fit
*around* the chain you already know how to build.


## 1. Setup

Install the packages we need. We're using `langchain` + `langchain-groq` —
the official LangChain integration for Groq — but the chain we build
later doesn't care which provider sits behind it.

**About "free":** Groq runs a genuine standing free tier — no credit card
required, no trial clock counting down. As of this writing it's rate
limited rather than credit-limited — exact request/token limits vary by
model and do change over time, so check the current numbers for your
model at [console.groq.com/docs/models](https://console.groq.com/docs/models)
if you want the precise figures.


In [ ]:
!pip install -qU langchain langchain-groq langchain-core


In [ ]:
import getpass
import os

# Never hardcode a key in the notebook. In Colab, prefer:
#   from google.colab import userdata
#   os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# Falling back to a manual prompt so this notebook runs standalone too.
# Get a free key at https://console.groq.com
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")


## 2. Four quick examples, before we build the real thing

Before we assemble the full travel assistant, let's see each of the four
core ideas on its own, in isolation, with the smallest possible example.
Nothing in this section is part of the final app — it's scratch space so
each idea is crystal clear before we combine all four.


### 2.1 Example — Connecting to the model

The simplest possible use of `ChatGroq`: create the connection object,
send one message, print the reply. No prompt template, no history, no
tools — just the "phone call" itself.


In [ ]:
from langchain_groq import ChatGroq

demo_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

response = demo_llm.invoke("In one sentence, what is LangChain?")
print(response.content)


`response` is more than just text — it's a structured object.
`response.content` is what we print above; later we'll also look at
`response.tool_calls`, which is empty here because we didn't give the
model any tools to ask for.


### 2.2 Example — Multi-turn conversation with memory

Same model, same question — "What can I see there?" — asked two different
ways: once with no history at all, and once with the earlier exchange
included. Watch how differently it answers.


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# No memory: the model only ever sees this one line.
no_memory_reply = demo_llm.invoke([
    HumanMessage(content="What can I see there?")
])
print("WITHOUT memory:\n", no_memory_reply.content)


In [ ]:
# With memory: we resend the earlier exchange first, then the new question.
history_example = [
    HumanMessage(content="I'm thinking of visiting Jaipur."),
    AIMessage(content="Nice choice! Jaipur is known as the Pink City."),
]

with_memory_reply = demo_llm.invoke(
    history_example + [HumanMessage(content="What can I see there?")]
)
print("WITH memory:\n", with_memory_reply.content)


Same exact final question both times. The only difference is whether
we included `history_example` in the list we sent. That's the entire
"memory" mechanism — there's nothing else to it.


### 2.3 Example — A chain

A tiny chain that has nothing to do with travel: it rewrites a sentence in
a given tone. This shows the shape — `prompt | model | parser` — without
tools or history distracting from it.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rephrase_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the user's sentence in a {tone} tone. "
               "Reply with only the rewritten sentence."),
    ("human", "{sentence}"),
])

rephrase_chain = rephrase_prompt | demo_llm | StrOutputParser()

result = rephrase_chain.invoke({
    "tone": "cheerful",
    "sentence": "The flight got delayed by three hours.",
})
print(result)


Three pieces, piped together: a prompt template that fills in
`{tone}` and `{sentence}`, the model, and `StrOutputParser()`, which just
unwraps the plain text so `rephrase_chain.invoke(...)` returns a string
instead of a full message object. Try changing `"tone"` to `"formal"` or
`"sarcastic"` and re-running — same chain, different input.


### 2.4 Example — Calling a tool

One tiny tool that has nothing to do with travel: adding two numbers.
Language models are notoriously unreliable at arithmetic on large
numbers — this is a good, honest demonstration of why handing off to real
code matters.


In [ ]:
from langchain_core.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """Add two whole numbers together and return the result."""
    return a + b

demo_llm_with_tool = demo_llm.bind_tools([add_numbers])

tool_response = demo_llm_with_tool.invoke("What is 482 plus 917?")
print("Model's tool request:", tool_response.tool_calls)


In [ ]:
# The model only *asked* for the calculation — it didn't run it.
# We run the real function ourselves, using the arguments the model gave us.
if tool_response.tool_calls:
    call = tool_response.tool_calls[0]
    result = add_numbers.invoke(call["args"])
    print(f"Tool actually computed: {result}")


Notice `tool_response.content` would likely be empty here — the
model didn't answer in words at all, it made a structured request instead.
`add_numbers.invoke(...)` is *our* code running, not the model's. That
request-then-execute split is tool calling, in full, with nothing else
going on.


With all four ideas seen in isolation, let's put them together and
build the real travel assistant.


## 3. The fake data

This is the assistant's entire "world." Nothing here calls a real API —
it's just Python dictionaries and a list, sitting in memory. Later, our
tools will simply read from these.


In [ ]:
FAKE_WEATHER_DB = {
    "Delhi": {"condition": "Hot and sunny", "temp_c": 36, "rain_chance": 20},
    "Jaipur": {"condition": "Sunny", "temp_c": 34, "rain_chance": 10},
    "Mumbai": {"condition": "Heavy clouds", "temp_c": 29, "rain_chance": 75},
    "Bengaluru": {"condition": "Pleasant with light rain", "temp_c": 25, "rain_chance": 55},
    "Goa": {"condition": "Cloudy", "temp_c": 28, "rain_chance": 65},
}

FAKE_ATTRACTIONS_DB = {
    "Delhi": [
        {"name": "India Gate", "type": "Landmark", "time_needed": "1 hour"},
        {"name": "Humayun's Tomb", "type": "Heritage", "time_needed": "2 hours"},
    ],
    "Jaipur": [
        {"name": "Amer Fort", "type": "Heritage", "time_needed": "3 hours"},
        {"name": "Hawa Mahal", "type": "Landmark", "time_needed": "1 hour"},
    ],
    "Mumbai": [
        {"name": "Gateway of India", "type": "Landmark", "time_needed": "1 hour"},
        {"name": "Marine Drive", "type": "Leisure", "time_needed": "2 hours"},
    ],
    "Bengaluru": [
        {"name": "Cubbon Park", "type": "Nature", "time_needed": "2 hours"},
        {"name": "Bangalore Palace", "type": "Heritage", "time_needed": "2 hours"},
    ],
    "Goa": [
        {"name": "Basilica of Bom Jesus", "type": "Heritage", "time_needed": "2 hours"},
        {"name": "Baga Beach", "type": "Leisure", "time_needed": "3 hours"},
    ],
}

FAKE_FLIGHTS_DB = [
    {"flight": "AI-101", "from": "Delhi", "to": "Jaipur", "departure": "08:10", "price_inr": 4200, "seats": 6},
    {"flight": "6E-221", "from": "Delhi", "to": "Jaipur", "departure": "18:45", "price_inr": 3900, "seats": 3},
    {"flight": "AI-305", "from": "Delhi", "to": "Mumbai", "departure": "09:30", "price_inr": 6100, "seats": 8},
    {"flight": "UK-822", "from": "Mumbai", "to": "Goa", "departure": "13:20", "price_inr": 3500, "seats": 4},
    {"flight": "6E-517", "from": "Bengaluru", "to": "Goa", "departure": "07:40", "price_inr": 3200, "seats": 5},
    {"flight": "AI-640", "from": "Jaipur", "to": "Bengaluru", "departure": "16:15", "price_inr": 7200, "seats": 2},
]


## 4. Turning the data into *tools*

A **tool** is just a normal Python function — but wrapped so the model can
see its name, its description, and what arguments it takes, *without* us
writing that description by hand. The `@tool` decorator reads the docstring
and type hints and builds that description for us.

The model itself never runs this code. It only ever says, in effect,
*"call `get_weather` with city='Jaipur'"* — a structured request. Our code
is the one that actually executes the function and hands the result back.
That separation — model decides, our code executes — is the whole idea
behind tool-calling.


In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the current weather for an Indian city (Delhi, Jaipur, Mumbai, Bengaluru, or Goa)."""
    data = FAKE_WEATHER_DB.get(city.title())
    if not data:
        return f"No weather data available for {city}."
    return (f"{city.title()}: {data['condition']}, {data['temp_c']}°C, "
            f"{data['rain_chance']}% chance of rain.")


@tool
def get_attractions(city: str) -> str:
    """List top attractions in an Indian city (Delhi, Jaipur, Mumbai, Bengaluru, or Goa)."""
    spots = FAKE_ATTRACTIONS_DB.get(city.title())
    if not spots:
        return f"No attraction data available for {city}."
    lines = [f"- {s['name']} ({s['type']}, ~{s['time_needed']})" for s in spots]
    return f"Top attractions in {city.title()}:\n" + "\n".join(lines)


@tool
def search_flights(from_city: str, to_city: str) -> str:
    """Search available flights between two Indian cities by name, e.g. from_city='Delhi', to_city='Jaipur'."""
    matches = [
        f for f in FAKE_FLIGHTS_DB
        if f["from"].lower() == from_city.lower() and f["to"].lower() == to_city.lower()
    ]
    if not matches:
        return f"No flights found from {from_city.title()} to {to_city.title()}."
    lines = [
        f"- {f['flight']} departs {f['departure']}, ₹{f['price_inr']}, {f['seats']} seats left"
        for f in matches
    ]
    return f"Flights from {from_city.title()} to {to_city.title()}:\n" + "\n".join(lines)


tools = [get_weather, get_attractions, search_flights]
tools_by_name = {t.name: t for t in tools}


## 5. The model — and *binding* the tools to it

`bind_tools` doesn't make the model able to run code. All it does is tell
the model, on every call, "here are three functions you're allowed to ask
for, and here's what each one needs as input." The model's reply will
either be plain text, or a request to call one (or more) of these tools.


In [ ]:
from langchain_groq import ChatGroq

# openai/gpt-oss-120b is the right default right now: it's on Groq's free
# developer tier (llama-3.3-70b-versatile, which older tutorials use, has
# since moved to Groq's Enterprise-only tier and will 404 on a free key).
# Groq's model lineup changes fairly often — if this model ever stops
# working, check the current free-tier list at
# https://console.groq.com/docs/models before assuming your code is wrong.
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
llm_with_tools = llm.bind_tools(tools)


## 6. The chain: prompt + model, with room for history

This is the "chain" piece: a prompt template piped into the tool-bound
model. Notice the `MessagesPlaceholder("history")` — that's the slot where
we'll inject everything said earlier in the conversation, every time we
call the chain. The chain itself has no memory; *we* supply the memory as
input each time.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant for a demo company. "
               "Use the available tools when the user asks about weather, "
               "attractions, or flights. Keep answers concise and friendly."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

chain = prompt | llm_with_tools


## 7. Carrying conversation history across turns

This is a plain Python list of messages that we grow after every turn. It's
deliberately simple so you can *see* exactly what the model is being shown
each time — no hidden memory object.


In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

conversation_history = []


## 8. Wiring it together: one full turn

Here's the actual flow for a single user message:

1. Send `{input}` plus the running `history` through the chain.
2. Look at what came back. If the model asked to call a tool, run that
   tool for real with our own code, and feed the *result* back to the
   model as a `ToolMessage` so it can turn raw data into a natural reply.
   (A model can ask for more than one tool call in the same turn — we
   handle all of them before asking the model to respond again.)
3. If the model just replied in plain text, that *is* the answer — no
   tool needed.
4. Whatever happened, append the new messages to `conversation_history` so
   the next turn can refer back to them.


In [ ]:
def chat(user_input: str) -> str:
    response = chain.invoke({
        "input": user_input,
        "history": conversation_history,
    })

    # Case 1: the model wants to call one or more tools first.
    if response.tool_calls:
        tool_messages = []
        for call in response.tool_calls:
            tool_fn = tools_by_name[call["name"]]
            result = tool_fn.invoke(call["args"])
            tool_messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

        # Give the model the tool result so it can phrase a real answer.
        follow_up = chain.invoke({
            "input": user_input,
            "history": conversation_history + [response] + tool_messages,
        })
        final_text = follow_up.content

        conversation_history.extend([
            HumanMessage(content=user_input),
            response,
            *tool_messages,
            follow_up,
        ])
        return final_text

    # Case 2: plain text answer, no tool needed.
    conversation_history.extend([
        HumanMessage(content=user_input),
        response,
    ])
    return response.content


## 9. Try a multi-turn conversation

Watch how the second and third questions lean on context from earlier
turns — "there" and "that trip" are never re-explained by the user.


In [ ]:
print(chat("What's the weather like in Jaipur right now?"))


In [ ]:
print(chat("What can I see there while I'm visiting?"))


In [ ]:
print(chat("Great — are there any flights from Delhi to there tomorrow?"))


In [ ]:
print(chat("And what's the weather looking like in Goa, in case I extend the trip?"))


## 10. Peek at the history (optional, for the class)

This is a good moment to show learners that "memory" isn't magic — it's
just this list, growing turn by turn, and getting re-sent every time.


In [ ]:
for m in conversation_history:
    role = m.__class__.__name__
    content = getattr(m, "content", "") or (m.tool_calls if hasattr(m, "tool_calls") else "")
    print(f"{role}: {content}")


## 11. Where to go from here

- Swap the manual `chat()` loop for LangChain's `RunnableWithMessageHistory`
  once learners are comfortable with what it's doing under the hood.
- Swap `ChatGroq` for any other tool-calling chat model (OpenAI, Anthropic,
  Google Gemini, etc.) — the tool definitions and chain don't change, only
  this one line.
- Add more tools (e.g. hotel search) by writing another `@tool` function
  and adding it to the `tools` list — nothing else in the chain changes.
